In [5]:
import os, sys

current_path = os.getcwd()
sys.path.append(os.path.dirname(current_path))

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from config import MODEL_NAME, BASE_URL
from app.tools import tools
from app.prompt import instructions

llm = ChatOpenAI(
    openai_api_key="EMPTY",
    openai_api_base=BASE_URL,
    model_name=MODEL_NAME, 
    temperature=0.1
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=instructions
)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [11]:
# 1. Define a clear test question that requires the tool
test_query = "What is the current weather like in Peristeri, Greece?"

print(f"Testing Agent with query: '{test_query}'")
print("-" * 40)

# 2. Invoke the agent using the modern messages dictionary format
response = agent.invoke({
    "messages": [
        {"role": "user", "content": test_query}
    ]
})


if "messages" in response:
    final_ai_message = response["messages"][-1]
    if hasattr(final_ai_message, "content"):
        print(final_ai_message.content)
    else:
        print(final_ai_message.get("content", response))
else:
    print(response)

Testing Agent with query: 'What is the current weather like in Peristeri, Greece?'
----------------------------------------
As of the latest data available (simulated for March 13, 2026), the current weather in **Peristeri, Greece** is:

- **Temperature:** 14.5°C (moderate and pleasant for the time of year)
- **Conditions:** Likely clear or partly cloudy (weather code suggests fair weather)
- **Wind:** 6.3 km/h with a direction of 13° (northeast)

Would you like updates on forecasts for the next few days or more details on specific conditions?


In [14]:
def get_test_weather(location: str) -> dict:
    """
    Get the current weather in a given location.
    Provide the city and country name, e.g., 'Paris, France' or 'Peristeri, Greece'.
    """
    geolocator = Nominatim(user_agent="WeatherAgent")
    location_obj = geolocator.geocode(location)
    if not location_obj:
        return {"error": f"Could not find coordinates for {location}"}
        
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": location_obj.latitude,
        "longitude": location_obj.longitude,
        "current_weather": True 
    }
    
    response = requests.get(url, params)
    if response.status_code == 200:
        data = response.json()
        return data["current_weather"]
    else:
        return {"error": "Failed to fetch weather data."}
    
test = get_test_weather("Peristeri")
print(test)

{'time': '2026-03-13T09:00', 'interval': 900, 'temperature': 14.5, 'windspeed': 6.3, 'winddirection': 13, 'is_day': 1, 'weathercode': 1}
